# MSC Figure 2 — fully reproducible recomputation

**No Google Drive required.** Upload two files and run every cell top to bottom. The
notebook ends by writing a single ZIP bundle to your computer.

## What you upload

| File | Role |
|---|---|
| `msc_model_feedback.py` | the MSC simulator (`simulate_one`, `sample_params`, `fit_binary_observation_model`, ...) |
| `config_msc_feedback.yaml` | the 27 parameter-sampling intervals and fixed settings |

If your model file is still called `msc_model_feedback_used.py`, the notebook renames it
automatically — the original runner imports the module under the name `msc_model_feedback`.

## What this notebook computes, and where it appears in the manuscript

| Output | Manuscript location |
|---|---|
| `mismatch_binned_docking_probability.csv` | **Figure 2A** — docking probability across 12 quantile bins of material mismatch d² |
| `msc_feedback_binary_observation_model.csv` | **Figure 2B** — logistic observation model, coefficients for T, M, d² |
| `figure2_paper_numbers.csv` | **Figure 2C** and abstract — encounter→docking→productive-capture percentages |
| `msc_feedback_nullmodel_summary.csv` | **Figure 3** — six-variant comparison (also produced here) |
| `FigS_panelA_kernel_consistency.csv` | **new supplementary panel** — simulated curve vs the analytic Gaussian-kernel expectation |
| `FigS_panelA_stratified_active.csv` | **new supplementary panel** — Panel A split into active vs inactive synapses |
| `tautology_binned_by_variant.csv` | **tautology panel** — d²-dependence in all six variants, not just full MSC |

## Canonical target

The archived `Figure2A/B_FINAL_2M` files correspond to **2,000,000 encounter observations**
and **6,079 docking events**, i.e. 400 parameter seeds × 5,000 retained observations per run.
Cell 6 verifies the fresh run against those exact values, including the first- and last-bin
docking probabilities and the d² coefficient. If the checks pass, the run is bit-identical.

## Two passes, deliberately separated

- **Pass 1** reproduces the published figure exactly. Encounter observations are recorded for
  `full_msc_feedback` only, matching the original runner.
- **Pass 2** is an additional run that records observations for **all six variants**. It writes to a
  separate directory so the canonical outputs can never be overwritten.


## 1 — Environment

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "statsmodels", "pyarrow"], check=False)

import hashlib, json, math, os, shutil, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

## 2 — Upload the model and the config

SHA-256 hashes of both input files are recorded and later written into the manifest, so any
figure in the bundle can be traced back to the exact code that produced it.

In [ ]:
from google.colab import files

WORK = Path("/content/msc_figure2")
WORK.mkdir(exist_ok=True)

print("Select msc_model_feedback.py (or msc_model_feedback_used.py) AND the .yaml config.")
uploaded = files.upload()
for name in uploaded:
    shutil.move(f"/content/{name}", WORK / name)

py_files = sorted(WORK.glob("*.py"))
yaml_files = sorted(list(WORK.glob("*.yaml")) + list(WORK.glob("*.yml")))
assert py_files, "no .py model file uploaded"
assert yaml_files, "no .yaml config uploaded"

MODEL = WORK / "msc_model_feedback.py"
if not MODEL.exists():
    py_files[0].rename(MODEL)
    print(f"renamed {py_files[0].name} -> msc_model_feedback.py")
CONFIG = yaml_files[0]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

INPUT_SHA = {p.name: sha256(p) for p in (MODEL, CONFIG)}
for k, v in INPUT_SHA.items():
    print(f"{v[:16]}  {k}")

os.chdir(WORK)
sys.path.insert(0, str(WORK))
import msc_model_feedback as msc
print("\nvariants found:")
for v, d in msc.MODEL_VARIANTS.items():
    print(f"  {v:32s} {d['label']}")

## 3 — Run configuration and output tree

```
figure2_output/
  01_inputs/                 verbatim copies of the model, config, run_config.json
  02_pass1_canonical/data/   the CSVs that reproduce the published Figure 2
  02_pass1_canonical/checkpoints/
  03_pass2_allvariants/data/ all-variant encounter observations (parquet) + per-variant fits
  04_derived/                kernel-consistency, stratified panel, paper-facing number table
  05_r_input/                exactly the files figure2.R expects, flat, ready to plot
  06_logs/                   verification report
  MANIFEST_figure2.json      run config + SHA-256 of every input and output
```

Keep `SMOKE = True` for a 4-seed test run first. Only set it to `False` once the verification
cell reports sensible output — the full run is 400 seeds × 6 variants = 2,400 simulations and
takes several hours.

In [ ]:
SMOKE = False          # <-- full canonical run

RUN = {
    "N_SEEDS": 400,              # Pass 1: 400 x 5000 = 2,000,000 (canonical)
    "N_SEEDS_PASS2": 200,        # Pass 2: 200 x 5000 = 1,000,000 per variant
    "SEED_BASE": 1000,
    "MAX_OBS_PER_RUN": 5000,
    "N_BINS": 12,
    "VARIANTS": list(msc.MODEL_VARIANTS),
    "OBS_VARIANT_PASS1": "full_msc_feedback",
    "CANONICAL_ENCOUNTERS": 2_000_000,
    "CANONICAL_DOCKS": 6079,
}

OUT = WORK / "figure2_output"
for sub in ["01_inputs", "02_pass1_canonical/data", "02_pass1_canonical/checkpoints",
            "03_pass2_allvariants/data", "04_derived", "05_r_input", "06_logs"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

shutil.copy2(MODEL, OUT / "01_inputs" / MODEL.name)
shutil.copy2(CONFIG, OUT / "01_inputs" / CONFIG.name)

cfg = msc.load_config(str(CONFIG))
(OUT / "01_inputs" / "run_config.json").write_text(json.dumps(RUN, indent=2))
print(json.dumps(RUN, indent=2))
print("\nsmoke run:", SMOKE)

## 4 — Pass 1: canonical simulation

This reproduces the seeding scheme of the original runner exactly. Each parameter-generating
seed defines one parameter realisation shared across all six variants; a deterministic
variant-specific offset then seeds initialisation, transport and event sampling. Comparisons
are therefore matched at the parameter-realisation level but do not impose identical event
histories across variants.

In [ ]:
def variant_offset(variant):
    """Deterministic per-variant seed offset, identical to run_msc_feedback_colab.py."""
    return sum((j + 1) * ord(ch) for j, ch in enumerate(variant)) % 10000

def pass1(n_seeds, checkpoint_every=20):
    metrics_rows, obs_frames, dock_frames = [], [], []
    t0 = time.time()
    for s in range(n_seeds):
        seed = RUN["SEED_BASE"] + s
        base = msc.sample_params(seed, cfg)
        base["max_obs_per_run"] = RUN["MAX_OBS_PER_RUN"]
        for variant in RUN["VARIANTS"]:
            record = (variant == RUN["OBS_VARIANT_PASS1"])
            out = msc.simulate_one(base, seed=seed * 100 + variant_offset(variant),
                                   variant_name=variant, record_obs=record)
            metrics_rows.append(out["metrics"])
            if record and not out["obs_df"].empty:
                obs_frames.append(out["obs_df"])
            if variant in {"full_msc_feedback", "msc_capture_only"} and not out["dock_df"].empty:
                dock_frames.append(out["dock_df"].head(20000))
        if (s + 1) % checkpoint_every == 0:
            pd.DataFrame(metrics_rows).to_csv(
                OUT / "02_pass1_canonical/checkpoints/metrics_checkpoint.csv", index=False)
            el = time.time() - t0
            print(f"[pass1] {s+1}/{n_seeds} seeds | {el/60:6.1f} min elapsed | "
                  f"ETA {el/(s+1)*(n_seeds-s-1)/60:6.1f} min", flush=True)
    return (pd.DataFrame(metrics_rows),
            pd.concat(obs_frames, ignore_index=True) if obs_frames else pd.DataFrame(),
            pd.concat(dock_frames, ignore_index=True) if dock_frames else pd.DataFrame())

metrics, obs, dock = pass1(4 if SMOKE else RUN["N_SEEDS"])
print(f"\nmetrics rows {len(metrics)} | encounter observations {len(obs):,} | docking events {len(dock):,}")

## 5 — Write the canonical Figure 2 / Figure 3 tables

Panel A uses 12 equal-count quantile bins of d², identical to `plot_mismatch()` in the original
runner. Panel B is the logistic observation model, which is a **diagnostic** that the simulated
events recover the encoded effect directions, not an independently fitted mechanistic model.

In [ ]:
D1 = OUT / "02_pass1_canonical/data"

summary   = msc.summarize_metrics(metrics)
pairs     = msc.paired_effects(metrics)
obs_model = msc.fit_binary_observation_model(obs)
mapping   = msc.equation_mapping()

tmp = obs.copy()
tmp["bin"] = pd.qcut(tmp["d2"], q=RUN["N_BINS"], duplicates="drop")
binned = (tmp.groupby("bin", observed=True)
             .agg(d2_mean=("d2", "mean"), p_dock=("dock", "mean"), n=("dock", "size"))
             .reset_index())

for df, name in [(metrics,   "msc_feedback_nullmodel_metrics.csv"),
                 (summary,   "msc_feedback_nullmodel_summary.csv"),
                 (pairs,     "msc_feedback_paired_effects.csv"),
                 (obs,       "msc_feedback_encounter_observations_sampled.csv"),
                 (dock,      "msc_feedback_docking_events_sampled.csv"),
                 (obs_model, "msc_feedback_binary_observation_model.csv"),
                 (mapping,   "equation_to_code_mapping_feedback.csv"),
                 (binned,    "mismatch_binned_docking_probability.csv")]:
    df.to_csv(D1 / name, index=False)
    print(f"{len(df):>10,} rows  {name}")

print("\nPanel A:")
print(binned.to_string(index=False))
print("\nPanel B:")
print(obs_model.to_string(index=False))

## 6 — Verification against the archived FINAL_2M figure data

These are the same guards that `figure2.R` applies, plus the coefficient check. On a smoke run
they are expected to fail on the count-based checks — that is the point of the smoke run.

In [ ]:
CANON = {"first_bin_p_dock": 0.00971998056003888,
         "last_bin_p_dock":  0.00048001344037633056,
         "beta_T":  3.42239207834849,
         "beta_M":  2.915147227029582,
         "beta_d2": -0.19045122928569383}

def coef(term):
    row = obs_model.loc[obs_model.term == term, "estimate"]
    return float(row.iloc[0]) if len(row) else float("nan")

checks = {
    "12 mismatch bins":            len(binned) == RUN["N_BINS"],
    "encounters == 2,000,000":     int(binned.n.sum()) == RUN["CANONICAL_ENCOUNTERS"],
    "docking events == 6,079":     int(obs_model.n_docks.iloc[0]) == RUN["CANONICAL_DOCKS"],
    "first-bin p_dock matches":    math.isclose(binned.p_dock.iloc[0],  CANON["first_bin_p_dock"], abs_tol=1e-12),
    "last-bin p_dock matches":     math.isclose(binned.p_dock.iloc[-1], CANON["last_bin_p_dock"],  abs_tol=1e-12),
    "beta_T matches":              math.isclose(coef("T"),  CANON["beta_T"],  abs_tol=1e-9),
    "beta_M matches":              math.isclose(coef("M"),  CANON["beta_M"],  abs_tol=1e-9),
    "beta_d2 matches":             math.isclose(coef("d2"), CANON["beta_d2"], abs_tol=1e-9),
}
report = pd.DataFrame({"check": list(checks), "passed": list(checks.values())})
report.to_csv(OUT / "06_logs/pass1_verification.csv", index=False)
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)

## 7 — Paper-facing numbers, with both estimators stated

The manuscript reports ~0.30 % of encounters docking and 66.59 % of docked events becoming
productive. The productive fraction depends on the estimator: the ratio of summed counts and
the mean of per-seed ratios are not equal. This cell prints both so the Methods can name the
one actually used.

In [ ]:
fm = metrics[metrics.variant == "full_msc_feedback"].copy()
n_enc = int(binned.n.sum())
n_dock = int(obs_model.n_docks.iloc[0])

ratio_of_means = fm.n_productive_docks.sum() / max(1, fm.n_docks.sum())
per_seed = fm.n_productive_docks / fm.n_docks.replace(0, np.nan)
mean_of_ratios = float(per_seed.dropna().mean())

slope, intercept = np.polyfit(binned.d2_mean[1:], np.log(binned.p_dock[1:]), 1)
ell_h_sampled = float(summary.loc[summary.variant == "full_msc_feedback", "ell_h_mean"].iloc[0])

table = pd.DataFrame([
    ("encounter opportunities",              n_enc,                                  "Methods / Fig 2C"),
    ("docking events",                       n_dock,                                 "Fig 2C"),
    ("P(dock | encounter), %",               round(n_dock / n_enc * 100, 4),         "abstract: ~0.30%"),
    ("P(productive | dock) %, ratio of means", round(ratio_of_means * 100, 2),       "abstract: 66.59%"),
    ("P(productive | dock) %, mean of ratios", round(mean_of_ratios * 100, 2),       "alternative estimator"),
    ("GLM beta_d2",                          round(coef("d2"), 4),                   "Fig 2B"),
    ("binned log-linear slope",              round(float(slope), 4),                 "kernel check"),
    ("implied ell_h from slope",             round(math.sqrt(-0.5 / slope), 3),      "should match Table S1"),
    ("sampled ell_h mean",                   round(ell_h_sampled, 3),                "Table S1"),
], columns=["quantity", "value", "manuscript_reference"])

table.to_csv(OUT / "04_derived/figure2_paper_numbers.csv", index=False)
print(table.to_string(index=False))

## 8 — Kernel consistency and the stratified Panel A

The docking hazard contains a Gaussian compatibility kernel, so log P(dock) must be linear in
d² with slope −0.5/ℓ_h². Two independent estimates of the same kernel width — the binned slope
and the logistic coefficient — should agree, and both should match the sampled ℓ_h.

The lowest-d² bin is expected to sit **above** the kernel prediction, because the active synapse
has its target vector at the origin and the compatible granule population is centred there, so
that bin is enriched for encounters at elevated T and M. The stratified table separates active
from inactive synapses and isolates the pure material effect.

In [ ]:
kern = binned.copy()
kern["kernel_pred"] = np.exp(intercept + slope * kern.d2_mean)
kern["obs_over_pred"] = kern.p_dock / kern.kernel_pred
r2 = float(np.corrcoef(kern.d2_mean[1:], np.log(kern.p_dock[1:]))[0, 1] ** 2)
kern.to_csv(OUT / "04_derived/FigS_panelA_kernel_consistency.csv", index=False)

print(f"log-linearity R^2 (bins 2-12): {r2:.3f}")
print(f"bin-1 excess over kernel prediction: {kern.obs_over_pred.iloc[0]:.2f}x\n")
print(kern[["d2_mean", "p_dock", "kernel_pred", "obs_over_pred", "n"]].round(6).to_string(index=False))

strat = obs.copy()
strat["bin"] = pd.qcut(strat["d2"], q=RUN["N_BINS"], duplicates="drop")
strat = (strat.groupby(["active_synapse", "bin"], observed=True)
              .agg(d2_mean=("d2", "mean"), p_dock=("dock", "mean"), n=("dock", "size"))
              .reset_index())
strat.to_csv(OUT / "04_derived/FigS_panelA_stratified_active.csv", index=False)
print("\nstratified observation counts:")
print(strat.groupby("active_synapse").n.sum().to_string())

## 9 — Pass 2: all six variants

The original runner records encounter observations only for `full_msc_feedback`, which is why the
tautology panel currently contains a single variant. This pass records all six, so the claim that
only material-aware variants show d²-dependence can actually be plotted. Output goes to a
separate directory; the canonical Pass 1 files are untouched.

In [ ]:
def pass2(n_seeds, checkpoint_every=20):
    frames, t0 = [], time.time()
    for s in range(n_seeds):
        seed = RUN["SEED_BASE"] + s
        base = msc.sample_params(seed, cfg)
        base["max_obs_per_run"] = RUN["MAX_OBS_PER_RUN"]
        for variant in RUN["VARIANTS"]:
            out = msc.simulate_one(base, seed=seed * 100 + variant_offset(variant),
                                   variant_name=variant, record_obs=True)
            if not out["obs_df"].empty:
                frames.append(out["obs_df"])
        if (s + 1) % checkpoint_every == 0:
            el = time.time() - t0
            print(f"[pass2] {s+1}/{n_seeds} seeds | {el/60:6.1f} min | "
                  f"ETA {el/(s+1)*(n_seeds-s-1)/60:6.1f} min", flush=True)
    return pd.concat(frames, ignore_index=True)

obs_all = pass2(4 if SMOKE else RUN["N_SEEDS_PASS2"])
D2 = OUT / "03_pass2_allvariants/data"
obs_all.to_parquet(D2 / "encounter_observations_all_variants.parquet", index=False)

curves = []
for variant, grp in obs_all.groupby("variant"):
    g = grp.copy()
    g["bin"] = pd.qcut(g["d2"], q=RUN["N_BINS"], duplicates="drop")
    t = (g.groupby("bin", observed=True)
          .agg(d2_mean=("d2", "mean"), p_dock=("dock", "mean"), n=("dock", "size"))
          .reset_index())
    t.insert(0, "variant", variant)
    curves.append(t)
taut = pd.concat(curves, ignore_index=True)
taut.to_csv(D2 / "tautology_binned_by_variant.csv", index=False)

fits = []
for variant, grp in obs_all.groupby("variant"):
    f = msc.fit_binary_observation_model(grp)
    f.insert(0, "variant", variant)
    fits.append(f)
pd.concat(fits, ignore_index=True).to_csv(D2 / "logit_per_variant.csv", index=False)

print("\nobservations per variant:")
print(obs_all.groupby("variant").size().to_string())

## 10 — Stage the flat inputs for the R plotting script

`05_r_input/` contains exactly the filenames `figure2.R` expects, so in R you only point
`INPUT_DIR` at this folder. Note that `figure2.R` currently hard-codes an absolute desktop path
and calls `stop()` when the encounter total is not 2,000,000 — change those guards to warnings
if you plot a smoke run.

In [ ]:
R_IN = OUT / "05_r_input"
staged = [(D1, "mismatch_binned_docking_probability.csv"),
          (D1, "msc_feedback_binary_observation_model.csv"),
          (D1, "msc_feedback_nullmodel_summary.csv"),
          (D1, "msc_feedback_paired_effects.csv"),
          (OUT / "04_derived", "FigS_panelA_kernel_consistency.csv"),
          (OUT / "04_derived", "FigS_panelA_stratified_active.csv"),
          (OUT / "04_derived", "figure2_paper_numbers.csv"),
          (D2, "tautology_binned_by_variant.csv"),
          (D2, "logit_per_variant.csv")]
for src, name in staged:
    shutil.copy2(src / name, R_IN / name)
    print(f"staged  {name}")

## 11 — Manifest, README and ZIP download

The manifest records the run configuration, the verification outcome and a SHA-256 hash of every
input and output file. Archive the ZIP as-is; it is the reproducibility unit for Figure 2.

In [ ]:
readme = f"""# MSC Figure 2 — reproducible bundle

Generated {pd.Timestamp.utcnow().isoformat()}Z
Smoke run: {SMOKE}   Seeds: {4 if SMOKE else RUN['N_SEEDS']}   max_obs_per_run: {RUN['MAX_OBS_PER_RUN']}

## Contents
- 01_inputs/                model, config and run_config.json actually used
- 02_pass1_canonical/data/  Figure 2A, 2B, 2C and Figure 3 source tables
- 03_pass2_allvariants/data/ all-variant encounter observations and per-variant logistic fits
- 04_derived/               kernel-consistency table, stratified Panel A, paper number table
- 05_r_input/               flat inputs for figure2.R
- 06_logs/                  verification of this run against the archived FINAL_2M values
- MANIFEST_figure2.json     run config, verification result, SHA-256 of every file

## Plotting
In figure2.R set INPUT_DIR to the extracted 05_r_input folder.

## Canonical reference
2,000,000 encounter observations, 6,079 docking events,
400 parameter seeds x 5,000 retained observations per run.
"""
(OUT / "README.md").write_text(readme)

manifest = {
    "figure": "Figure 2",
    "generated_utc": pd.Timestamp.utcnow().isoformat(),
    "smoke_run": SMOKE,
    "seeds_pass2": 4 if SMOKE else RUN["N_SEEDS_PASS2"],
    "run_config": RUN,
    "canonical_reference": CANON,
    "verification": {k: bool(v) for k, v in checks.items()},
    "input_sha256": INPUT_SHA,
    "output_sha256": {str(p.relative_to(OUT)): sha256(p)
                      for p in sorted(OUT.rglob("*")) if p.is_file()},
}
(OUT / "MANIFEST_figure2.json").write_text(json.dumps(manifest, indent=2))

zip_path = Path("/content/MSC_Figure2_reproducible_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUT.parent))

print(f"{zip_path.name}  {zip_path.stat().st_size/1e6:.2f} MB")
print(f"files in bundle: {sum(1 for p in OUT.rglob('*') if p.is_file())}")
files.download(str(zip_path))